In [1]:
from lxml import etree
from pathlib import Path

In [2]:
NS = {"tei": "http://www.tei-c.org/ns/1.0"}

tree = etree.parse(Path("sources/20240818.xml"))

interesting = {
    "text",
    "front",
    "body",
    "back",
    "div",
    "listWit",
    "listBibl",
}

for el in tree.xpath("//tei:text//*", namespaces=NS):
    name = etree.QName(el).localname

    if name not in interesting:
        continue

    depth = sum(etree.QName(a).localname in interesting for a in el.iterancestors())

    attrs = []

    if el.get("type"):
        attrs.append(f'type="{el.get("type")}"')

    xml_id = el.get("{http://www.w3.org/XML/1998/namespace}id")
    if xml_id:
        attrs.append(f'xml:id="{xml_id}"')

    details = " " + " ".join(attrs) if attrs else ""

    print(f"{'  ' * depth}{name}{details}")

  front xml:id="front"
    div type="introduction" xml:id="preface"
      div type="section" xml:id="preface-section-1"
        div type="section" xml:id="preface-subsection-1.1"
        div type="section" xml:id="preface-subsection-1.2"
      div type="section" xml:id="preface-section-2"
        div type="section" xml:id="preface-subsection-2.1"
        div type="section" xml:id="preface-subsection-2.2"
          div type="section" xml:id="bibliography-manuscripts"
            listWit xml:id="α"
              listWit
              listWit
            listWit xml:id="κ"
              listWit
            listWit
              listWit
              listWit
          div type="section" xml:id="bibliography-early-editions"
            listWit xml:id="early-editions"
        div type="section" xml:id="preface-subsection-2.3"
        div type="section" xml:id="preface-subsection-2.4"
        div type="section" xml:id="preface-subsection-2.5"
        div type="section" xml:id="preface-subsect

In [3]:
import pandas as pd
rows = []

for app in tree.xpath("//tei:app", namespaces=NS):
    app_id = app.get("{http://www.w3.org/XML/1998/namespace}id")

    for reading in app.xpath("./tei:lem | ./tei:rdg", namespaces=NS):
        rows.append(
            {
                "app_id": app_id,
                "type": etree.QName(reading).localname,
                "reading": " ".join(reading.itertext()).strip(),
                "wit": reading.get("wit"),
                "source": reading.get("source"),
            }
        )

df = pd.DataFrame(rows)

In [4]:
df

,app_id,type,reading,wit,source
0,NaN,lem,Incipit,#γ #Nu #Px #Vm #Vo,None
1,NaN,rdg,Incipit decimanona particula.,#m #v,None
2,NaN,rdg,Incipit decimanona \n partic...,#Pz,None
3,NaN,rdg,xix,#Ea,None
4,NaN,rdg,Sequitur xviiii.,#Py,None
...,...,...,...,...,...
11497,NaN,rdg,sequitur,#Vz,None
11498,NaN,rdg,amen. Deo\n gratias amen,#Vb,None
11499,NaN,rdg,Explicit decimanona particula.,#Pz #m #v,None
11500,NaN,rdg,Explicit particula 19a deo gratias,#Va,None


In [5]:
df["wit"].value_counts()

wit
#Py                             478
#Va                             410
#Px                             400
#Pz                             364
#π                              339
                               ... 
#κ #Gu #Pp #Pz #Vo #m #v          1
#Pu #Vb                           1
#γ #Ea #Ma #Px #Vo #Vz #m #v      1
#Gu #Pp #Pu #Py #Pz #Vm           1
#Pu #Pz #Vb #Vz                   1
Name: count, Length: 2419, dtype: int64

In [6]:
df["witnesses"] = df["wit"].fillna("").str.split()

In [7]:
df["n_witnesses"] = df["witnesses"].str.len()

In [8]:
witness_report = df.sort_values("n_witnesses", ascending=False)[
    ["app_id", "type", "reading", "witnesses", "n_witnesses"]
]

In [9]:
witness_report["witness_set"] = witness_report["witnesses"].apply(frozenset)

In [10]:
witness_report["witness_set"].value_counts().head(50)

witness_set
frozenset({#Py})                                   478
frozenset({#Va})                                   410
frozenset({#Px})                                   400
frozenset({#Pz})                                   364
frozenset({#π})                                    339
frozenset({#Ea})                                   298
frozenset({#v, #m})                                296
frozenset({#Gu})                                   247
frozenset({#Vz})                                   204
frozenset({#Pq, #Gu})                              185
frozenset({#Pp})                                   179
frozenset({#Vb})                                   168
frozenset({#Ma})                                   164
frozenset({#m, #v, #γ})                            152
frozenset({#Pp, #Gu, #Pu})                         146
frozenset({#m, #v, #Va})                           143
frozenset({#Vm})                                   143
frozenset({#κ})                                    12

In [11]:
witness_report["type"].value_counts()

type
rdg    7195
lem    4307
Name: count, dtype: int64

In [12]:
groups = {}

for group in tree.xpath("//tei:listWit[@xml:id]", namespaces=NS):
    group_id = group.get("{http://www.w3.org/XML/1998/namespace}id")

    witnesses = group.xpath(".//tei:witness/@xml:id", namespaces=NS)

    groups[group_id] = frozenset(witnesses)

In [13]:
groups

{'α': frozenset({'Ea', 'Nu', 'Py', 'Va', 'Vb', 'β', 'γ'}),
 'κ': frozenset({'Ma', 'Mb', 'Px', 'Vm', 'Vz', 'λ'}),
 'early-editions': frozenset({'m', 'v'})}

In [14]:
mss_groups = {
    "α": frozenset({"Ea", "Nu", "Py", "Va", "Vb"}),
    "β": frozenset({"Nu", "Py"}),
    "γ": frozenset({"Va", "Vb"}),
    "κ": frozenset({"Ma", "Mb", "Px", "Vm", "Vz"}),
    "λ": frozenset({"Ma", "Vm", "Vz", "Mb"}),
    "j": frozenset({"Gu", "Pp", "Pu", "Pz", "Vo", "Ca", "Pa", "Pq", "Sc", "Vc"}),
    "π": frozenset({"Gu", "Pp", "Pu", "Pz", "Vo"})
}

In [15]:
def match_group(witness_set):
    # Try to match the witness set to one of the known groups in mss_groups
    for group_name, group_witnesses in mss_groups.items():
        if witness_set == group_witnesses:
            return group_name
        else:
            # Check if the witness set is a subset of the group witnesses
            if witness_set.issubset(group_witnesses):
                return f"{group_name} (subset)"

In [16]:
witness_report.columns

Index(['app_id', 'type', 'reading', 'witnesses', 'n_witnesses', 'witness_set'], dtype='str')

In [17]:
witness_report["witness_groups_match"] = witness_report["witness_set"].apply(match_group)

In [18]:
witness_report["witness_groups_match"].value_counts()

witness_groups_match
α (subset)    56
Name: count, dtype: int64

In [19]:
def match_group(witness_set):
    # Look for exact matches first
    for group_name, group_witnesses in mss_groups.items():
        if witness_set == group_witnesses:
            return group_name

In [20]:
witness_report["witness_groups_match"] = witness_report["witness_set"].apply(
    match_group
)

In [21]:
witness_report["witness_groups_match"].value_counts()

Series([], Name: count, dtype: int64)

In [22]:
def expand_groups(witness_set):
    expanded = set()

    for witness in witness_set:
        witness = witness.lstrip("#")
        if witness in mss_groups:
            expanded.update(mss_groups[witness])
        else:
            expanded.add(witness)

    return frozenset(expanded)

In [23]:
witness_report["expanded_witness_set"] = witness_report["witness_set"].apply(
    expand_groups
)

In [24]:
witness_report["expanded_witness_groups_match"] = witness_report["expanded_witness_set"].apply(
    match_group
)

In [25]:
witness_report["expanded_witness_groups_match"].value_counts()

expanded_witness_groups_match
π    339
κ    121
λ     64
β     30
γ     24
α      7
Name: count, dtype: int64

In [26]:
def group_already_used(row):
    if row["expanded_witness_groups_match"] is None:
        return False
    else:
        group = "#" + str(row["expanded_witness_groups_match"])
    return group in row["witness_set"]

witness_report["group_already_used"] = witness_report.apply(group_already_used, axis=1)

In [27]:
witness_report["group_already_used"].value_counts()

group_already_used
False    10917
True       585
Name: count, dtype: int64

In [28]:
def compare_to_groups(witness_set):
    results = {}

    for group_name, group_witnesses in mss_groups.items():
        missing = group_witnesses - witness_set
        extra = witness_set - group_witnesses

        results[group_name] = {
            "missing": missing,
            "extra": extra,
        }

    return results

In [29]:
witness_report["compare_to_groups"] = witness_report["expanded_witness_set"].apply(
    compare_to_groups
)

In [30]:
witness_report["compare_to_groups"].value_counts()

compare_to_groups
{'α': {'missing': frozenset({'Nu', 'Vb', 'Va', 'Ea'}), 'extra': frozenset({})}, 'β': {'missing': frozenset({'Nu'}), 'extra': frozenset({})}, 'γ': {'missing': frozenset({'Vb', 'Va'}), 'extra': frozenset({'Py'})}, 'κ': {'missing': frozenset({'Px', 'Mb', 'Vz', 'Ma', 'Vm'}), 'extra': frozenset({'Py'})}, 'λ': {'missing': frozenset({'Ma', 'Mb', 'Vm', 'Vz'}), 'extra': frozenset({'Py'})}, 'j': {'missing': frozenset({'Pu', 'Pz', 'Vc', 'Vo', 'Sc', 'Pp', 'Pq', 'Pa', 'Ca', 'Gu'}), 'extra': frozenset({'Py'})}, 'π': {'missing': frozenset({'Pu', 'Pz', 'Vo', 'Pp', 'Gu'}), 'extra': frozenset({'Py'})}}                                                                              478
{'α': {'missing': frozenset({'Nu', 'Vb', 'Py', 'Ea'}), 'extra': frozenset({})}, 'β': {'missing': frozenset({'Nu', 'Py'}), 'extra': frozenset({'Va'})}, 'γ': {'missing': frozenset({'Vb'}), 'extra': frozenset({})}, 'κ': {'missing': frozenset({'Px', 'Mb', 'Vz', 'Ma', 'Vm'}), 'extra': frozenset({'Va'})}, 'λ': {'m

In [31]:
def group_distance(witness_set, group_witnesses):
    missing = group_witnesses - witness_set
    extra = witness_set - group_witnesses

    return len(missing) + len(extra)

In [32]:
witness_report.columns

Index(['app_id', 'type', 'reading', 'witnesses', 'n_witnesses', 'witness_set',
       'witness_groups_match', 'expanded_witness_set',
       'expanded_witness_groups_match', 'group_already_used',
       'compare_to_groups'],
      dtype='str')

In [33]:
unique_expanded_witness_sets = witness_report["expanded_witness_set"].unique()

In [34]:
len(unique_expanded_witness_sets)

2331

In [35]:
for witness_set in unique_expanded_witness_sets:
    print(f"Witness set: {witness_set}")
    for group_name, group_witnesses in mss_groups.items():
        distance = group_distance(witness_set, group_witnesses)
        print(f"  Distance to group {group_name}: {distance}")

Witness set: frozenset({'Pz', 'Vo', 'Px', 'Ea', 'Vz', 'Pp', 'v', 'Ma', 'm', 'Vm', 'Nu', 'Gu'})
  Distance to group α: 13
  Distance to group β: 12
  Distance to group γ: 14
  Distance to group κ: 9
  Distance to group λ: 10
  Distance to group j: 14
  Distance to group π: 9
Witness set: frozenset({'Vz', 'Pp', 'v', 'Va', 'Pu', 'Pz', 'Vo', 'Px', 'Py', 'Ea', 'Mb', 'Vm', 'm', 'Nu', 'Vb', 'Gu'})
  Distance to group α: 11
  Distance to group β: 14
  Distance to group γ: 14
  Distance to group κ: 13
  Distance to group λ: 14
  Distance to group j: 16
  Distance to group π: 11
Witness set: frozenset({'Pu', 'Pz', 'Vo', 'Px', 'Py', 'Ea', 'Vz', 'Pp', 'v', 'm', 'Vm', 'Va'})
  Distance to group α: 11
  Distance to group β: 12
  Distance to group γ: 12
  Distance to group κ: 11
  Distance to group λ: 12
  Distance to group j: 14
  Distance to group π: 9
Witness set: frozenset({'Pu', 'Vo', 'Px', 'Py', 'Ea', 'Vz', 'Pp', 'v', 'm', 'Vm', 'Nu', 'Vb', 'Gu'})
  Distance to group α: 10
  Distance to group β

In [36]:
witness_report["expanded_witness_set"].value_counts().head(50)

expanded_witness_set
frozenset({Py})                                                              478
frozenset({Va})                                                              410
frozenset({Px})                                                              400
frozenset({Pz})                                                              364
frozenset({Pu, Pz, Pp, Vo, Gu})                                              339
frozenset({Ea})                                                              298
frozenset({v, m})                                                            297
frozenset({Gu})                                                              247
frozenset({Vz})                                                              204
frozenset({Pq, Gu})                                                          185
frozenset({Pp})                                                              179
frozenset({Vb})                                                              168
frozens

In [37]:
witness_report.columns

Index(['app_id', 'type', 'reading', 'witnesses', 'n_witnesses', 'witness_set',
       'witness_groups_match', 'expanded_witness_set',
       'expanded_witness_groups_match', 'group_already_used',
       'compare_to_groups'],
      dtype='str')

In [38]:
exact_matches = witness_report[witness_report["witness_groups_match"].notna()].copy()

In [39]:
exact_matches[["witness_set", "expanded_witness_set", "witness_groups_match"]].head(25)

,witness_set,expanded_witness_set,witness_groups_match


In [40]:
def contained_groups(witness_set):
    matches = []

    for group_name, group_witnesses in mss_groups.items():
        if group_witnesses.issubset(witness_set):
            matches.append(group_name)

    return matches

In [41]:
witness_report["contained_groups"] = witness_report["expanded_witness_set"].apply(
    contained_groups
)

In [42]:
witness_report['contained_groups'].value_counts()

contained_groups
[]                    6359
[π]                    709
[α, β, γ, κ, λ]        586
[κ, λ]                 512
[γ]                    494
[α, β, γ]              370
[γ, κ, λ]              343
[α, β, γ, π]           234
[β]                    224
[λ]                    176
[κ, λ, π]              170
[β, κ, λ]              167
[α, β, γ, λ]           155
[β, κ, λ, π]           133
[α, β, γ, κ, λ, π]     116
[γ, π]                 110
[β, γ, κ, λ]           108
[γ, κ, λ, π]            79
[β, π]                  69
[β, γ]                  61
[γ, λ]                  59
[α, β, γ, λ, π]         55
[β, λ]                  48
[γ, λ, π]               44
[λ, π]                  34
[β, γ, π]               25
[β, λ, π]               21
[β, γ, λ]               18
[β, γ, κ, λ, π]         13
[β, γ, λ, π]            10
Name: count, dtype: int64

In [43]:
def subgroup_already_used(row):
    if row["contained_groups"] is None:
        return False
    else:
        return [("#" + str(group)) in row["witness_set"] for group in row["contained_groups"]]

In [44]:
witness_report['subgroup_already_used'] = witness_report.apply(subgroup_already_used, axis=1)

In [45]:
witness_report['subgroup_already_used'].value_counts()

subgroup_already_used
[]                                          6359
[True]                                      1595
[True, False, False, True, False]            585
[True, False]                                514
[True, True, False]                          509
[True, False, False, True]                   384
[True, True]                                 377
[True, False, False]                         369
[True, True, False, True]                    211
[True, False, True]                          171
[True, False, False, True, False, True]      114
[True, True, True]                           108
[True, True, True, False]                    108
[True, False, False, True, True]              55
[True, True, True, False, True]               13
[True, True, True, True]                      10
[False]                                        8
[True, False, False, False]                    4
[False, True]                                  2
[True, True, False, False]                     

In [46]:
def unused_contained_groups(row):
    unused = []

    for group in row["contained_groups"]:
        if f"#{group}" not in row["witness_set"]:
            unused.append(group)

    return unused

In [47]:
witness_report["unused_contained_groups"] = witness_report.apply(
    unused_contained_groups, axis=1
)

In [48]:
witness_report["unused_contained_groups"].value_counts()

unused_contained_groups
[]              8449
[λ]             1523
[β, γ]           808
[β, γ, λ]        699
[β]                6
[π]                5
[β, γ, π]          4
[β, λ]             2
[λ, π]             1
[α]                1
[κ]                1
[β, γ, λ, π]       1
[γ]                1
[α, λ]             1
Name: count, dtype: int64

In [49]:
from itertools import combinations


def optimal_representation(witness_set):
    best_representation = set(witness_set)

    group_names = list(mss_groups)

    for n in range(1, len(group_names) + 1):
        for group_combo in combinations(group_names, n):

            covered = set()

            for group in group_combo:
                covered.update(mss_groups[group])

            # We can't use a group if it introduces witnesses
            # that aren't actually in the reading.
            if not covered.issubset(witness_set):
                continue

            # Anything not covered by groups must be represented
            # by its individual siglum.
            remaining = set(witness_set) - covered

            representation = set(group_combo) | remaining

            if len(representation) < len(best_representation):
                best_representation = representation

    return frozenset(best_representation)

In [50]:
witness_report["optimal_representation"] = witness_report["expanded_witness_set"].apply(
    optimal_representation
)

In [51]:
witness_report["actual_n_sigla"] = witness_report["witness_set"].apply(len)

In [52]:
witness_report["optimal_n_sigla"] = witness_report["optimal_representation"].apply(len)

In [53]:
witness_report["sigla_saved"] = (
    witness_report["actual_n_sigla"] - witness_report["optimal_n_sigla"]
)

In [54]:
witness_report.loc[
    witness_report["sigla_saved"] > 0,
    [
        "witness_set",
        "expanded_witness_set",
        "optimal_representation",
        "actual_n_sigla",
        "optimal_n_sigla",
        "sigla_saved",
    ],
].head(30)

,witness_set,expanded_witness_set,optimal_representation,actual_n_sigla,optimal_n_sigla,sigla_saved
126,"frozenset({#Pu, #Px, #Vz, #Pp, #Mb, #Vm, #m, #...","frozenset({Vz, Pp, v, Va, Pu, Pz, Vo, Px, Py, ...","frozenset({Px, Mb, Vz, v, π, Vm, m, α})",12,8,4
2282,"frozenset({#Pu, #Ea, #Nu, #Px, #Pp, #m, #v, #V...","frozenset({Pu, Pz, Vo, Px, Ea, Pp, v, Ma, m, N...","frozenset({Px, Ea, v, Ma, π, m, Nu, Va})",12,8,4
2139,"frozenset({#Pu, #Ea, #Px, #Vz, #Pp, #m, #Vm, #...","frozenset({Pu, Pz, Vo, Px, Ea, Vz, Pp, v, m, V...","frozenset({γ, Px, Ea, Vz, v, π, m, Vm})",12,8,4
5635,"frozenset({#Px, #Pu, #Ea, #Vz, #Pp, #m, #Vm, #...","frozenset({Pu, Pz, Vo, Px, Py, Ea, Vz, Pp, v, ...","frozenset({Pu, Pz, Vo, Px, Vz, Pp, v, m, Vm, α})",11,10,1
2189,"frozenset({#Pu, #Ea, #Px, #Py, #Pp, #m, #v, #V...","frozenset({Pu, Vo, Px, Py, Ea, Mb, Vz, Pp, v, ...","frozenset({Pu, Vo, γ, Py, Ea, Pp, κ, v, m, Gu})",11,10,1
696,"frozenset({#Pu, #Ea, #Nu, #Pp, #m, #v, #Vo, #G...","frozenset({Vz, Pp, v, Va, Pu, Pz, Vo, Px, Ea, ...","frozenset({κ, v, π, Nu, m, γ, Ea})",11,7,4
2327,"frozenset({#Pu, #Ea, #Nu, #Pp, #Pq, #Vm, #Vo, ...","frozenset({Pu, Pz, Vo, Ea, Pp, Ma, Pq, Vm, Nu,...","frozenset({Ma, π, Pq, Vm, Nu, Ea})",10,6,4
2283,"frozenset({#Px, #Py, #Nu, #Pp, #m, #v, #Vo, #G...","frozenset({Vo, Px, Py, Pp, v, Ma, m, Nu, Va, Gu})","frozenset({Vo, β, Px, Pp, v, Ma, m, Va, Gu})",10,9,1
7825,"frozenset({#Vb, #Pu, #Pp, #β, #m, #v, #λ, #Vo,...","frozenset({Pu, Pz, Vo, Py, Mb, Vz, Pp, v, Ma, ...","frozenset({Pu, Pz, λ, Vo, β, γ, Pp, v, m})",10,9,1
7413,"frozenset({#Pu, #Ea, #Pp, #β, #m, #v, #λ, #Vo,...","frozenset({Pu, Pz, Vo, Py, Ea, Mb, Vz, Pp, v, ...","frozenset({Pu, Pz, λ, Vo, Pp, v, m, α})",10,8,2


In [55]:
witness_report["sigla_saved"].min()

np.int64(0)

In [56]:
witness_report["sigla_saved"].value_counts()

sigla_saved
0    11450
1       38
4       11
2        2
3        1
Name: count, dtype: int64

In [57]:
witness_report.columns

Index(['app_id', 'type', 'reading', 'witnesses', 'n_witnesses', 'witness_set',
       'witness_groups_match', 'expanded_witness_set',
       'expanded_witness_groups_match', 'group_already_used',
       'compare_to_groups', 'contained_groups', 'subgroup_already_used',
       'unused_contained_groups', 'optimal_representation', 'actual_n_sigla',
       'optimal_n_sigla', 'sigla_saved'],
      dtype='str')

In [58]:
witness_report.groupby("type")["sigla_saved"].sum()

type
lem    82
rdg     7
Name: sigla_saved, dtype: int64

In [59]:
candidates = witness_report[witness_report["sigla_saved"] > 1].copy()

In [60]:
candidates.columns

Index(['app_id', 'type', 'reading', 'witnesses', 'n_witnesses', 'witness_set',
       'witness_groups_match', 'expanded_witness_set',
       'expanded_witness_groups_match', 'group_already_used',
       'compare_to_groups', 'contained_groups', 'subgroup_already_used',
       'unused_contained_groups', 'optimal_representation', 'actual_n_sigla',
       'optimal_n_sigla', 'sigla_saved'],
      dtype='str')

In [61]:
candidates_reduced = candidates[
    [
        "app_id",
        "type",
        "reading",
        "witnesses",
        "optimal_representation",
        "actual_n_sigla",
        "optimal_n_sigla",
        "sigla_saved"
    ]
]

In [62]:
candidates_reduced

,app_id,type,reading,witnesses,optimal_representation,actual_n_sigla,optimal_n_sigla,sigla_saved
126,NaN,lem,mundanam,"[#α, #Gu, #Mb, #Pp, #Pu, #Px, #Pz, #Vm, #Vo, #...","frozenset({Px, Mb, Vz, v, π, Vm, m, α})",12,8,4
2282,NaN,lem,dyaphonia,"[#Ea, #Gu, #Ma, #Nu, #Pp, #Pu, #Pz, #Px, #Va, ...","frozenset({Px, Ea, v, Ma, π, m, Nu, Va})",12,8,4
2139,NaN,lem,netem,"[#γ, #Ea, #Gu, #Pp, #Pu, #Px, #Pz, #Vm, #Vo, #...","frozenset({γ, Px, Ea, Vz, v, π, m, Vm})",12,8,4
696,NaN,lem,notam,"[#γ, #κ, #Ea, #Gu, #Nu, #Pp, #Pu, #Pz, #Vo, #m...","frozenset({κ, v, π, Nu, m, γ, Ea})",11,7,4
2327,NaN,lem,antiphonum,"[#Ea, #Gu, #Ma, #Nu, #Pp, #Pq, #Pu, #Pz, #Vm, ...","frozenset({Ma, π, Pq, Vm, Nu, Ea})",10,6,4
7413,NaN,lem,duabus,"[#β, #γ, #λ, #Ea, #Pp, #Pu, #Pz, #Vo, #m, #v]","frozenset({Pu, Pz, λ, Vo, Pp, v, m, α})",10,8,2
253,NaN,lem,ysidorum,"[#β, #Gu, #Ma, #Pp, #Pu, #Px, #Pz, #Vb, #Vm, #Vo]","frozenset({Ma, π, Vm, β, Px, Vb})",10,6,4
406,NaN,lem,consueto,"[#α, #Gu, #Pp, #Pu, #Px, #Pz, #Vo, #m, #v]","frozenset({v, π, m, Px, α})",9,5,4
3695,NaN,lem,648.,"[#α, #Gu, #Pp, #Pu, #Pz, #Vo, #m, #v]","frozenset({v, π, α, m})",8,4,4
2581,NaN,lem,vocibus et,"[#α, #Gu, #Pp, #Pu, #Pz, #Vo, #m, #v]","frozenset({v, π, α, m})",8,4,4


In [63]:
def unpack_optimal_representation(optimal_representation):
    unpacked = [ms for ms in optimal_representation]
    return unpacked

In [64]:
candidates_reduced["proposed"] = candidates_reduced["optimal_representation"].apply(unpack_optimal_representation)

In [65]:
candidates_reduced

,app_id,type,reading,witnesses,optimal_representation,actual_n_sigla,optimal_n_sigla,sigla_saved,proposed
126,NaN,lem,mundanam,"[#α, #Gu, #Mb, #Pp, #Pu, #Px, #Pz, #Vm, #Vo, #...","frozenset({Px, Mb, Vz, v, π, Vm, m, α})",12,8,4,"[Px, Mb, Vz, v, π, Vm, m, α]"
2282,NaN,lem,dyaphonia,"[#Ea, #Gu, #Ma, #Nu, #Pp, #Pu, #Pz, #Px, #Va, ...","frozenset({Px, Ea, v, Ma, π, m, Nu, Va})",12,8,4,"[Px, Ea, v, Ma, π, m, Nu, Va]"
2139,NaN,lem,netem,"[#γ, #Ea, #Gu, #Pp, #Pu, #Px, #Pz, #Vm, #Vo, #...","frozenset({γ, Px, Ea, Vz, v, π, m, Vm})",12,8,4,"[γ, Px, Ea, Vz, v, π, m, Vm]"
696,NaN,lem,notam,"[#γ, #κ, #Ea, #Gu, #Nu, #Pp, #Pu, #Pz, #Vo, #m...","frozenset({κ, v, π, Nu, m, γ, Ea})",11,7,4,"[κ, v, π, Nu, m, γ, Ea]"
2327,NaN,lem,antiphonum,"[#Ea, #Gu, #Ma, #Nu, #Pp, #Pq, #Pu, #Pz, #Vm, ...","frozenset({Ma, π, Pq, Vm, Nu, Ea})",10,6,4,"[Ma, π, Pq, Vm, Nu, Ea]"
7413,NaN,lem,duabus,"[#β, #γ, #λ, #Ea, #Pp, #Pu, #Pz, #Vo, #m, #v]","frozenset({Pu, Pz, λ, Vo, Pp, v, m, α})",10,8,2,"[Pu, Pz, λ, Vo, Pp, v, m, α]"
253,NaN,lem,ysidorum,"[#β, #Gu, #Ma, #Pp, #Pu, #Px, #Pz, #Vb, #Vm, #Vo]","frozenset({Ma, π, Vm, β, Px, Vb})",10,6,4,"[Ma, π, Vm, β, Px, Vb]"
406,NaN,lem,consueto,"[#α, #Gu, #Pp, #Pu, #Px, #Pz, #Vo, #m, #v]","frozenset({v, π, m, Px, α})",9,5,4,"[v, π, m, Px, α]"
3695,NaN,lem,648.,"[#α, #Gu, #Pp, #Pu, #Pz, #Vo, #m, #v]","frozenset({v, π, α, m})",8,4,4,"[v, π, α, m]"
2581,NaN,lem,vocibus et,"[#α, #Gu, #Pp, #Pu, #Pz, #Vo, #m, #v]","frozenset({v, π, α, m})",8,4,4,"[v, π, α, m]"


In [66]:
candidates_reduced["witness_no_hash"] = candidates_reduced["witnesses"].apply(lambda x: [w.lstrip("#") for w in x])

In [67]:
candidates_reduced

,app_id,type,reading,witnesses,optimal_representation,actual_n_sigla,optimal_n_sigla,sigla_saved,proposed,witness_no_hash
126,NaN,lem,mundanam,"[#α, #Gu, #Mb, #Pp, #Pu, #Px, #Pz, #Vm, #Vo, #...","frozenset({Px, Mb, Vz, v, π, Vm, m, α})",12,8,4,"[Px, Mb, Vz, v, π, Vm, m, α]","[α, Gu, Mb, Pp, Pu, Px, Pz, Vm, Vo, Vz, m, v]"
2282,NaN,lem,dyaphonia,"[#Ea, #Gu, #Ma, #Nu, #Pp, #Pu, #Pz, #Px, #Va, ...","frozenset({Px, Ea, v, Ma, π, m, Nu, Va})",12,8,4,"[Px, Ea, v, Ma, π, m, Nu, Va]","[Ea, Gu, Ma, Nu, Pp, Pu, Pz, Px, Va, Vo, m, v]"
2139,NaN,lem,netem,"[#γ, #Ea, #Gu, #Pp, #Pu, #Px, #Pz, #Vm, #Vo, #...","frozenset({γ, Px, Ea, Vz, v, π, m, Vm})",12,8,4,"[γ, Px, Ea, Vz, v, π, m, Vm]","[γ, Ea, Gu, Pp, Pu, Px, Pz, Vm, Vo, Vz, m, v]"
696,NaN,lem,notam,"[#γ, #κ, #Ea, #Gu, #Nu, #Pp, #Pu, #Pz, #Vo, #m...","frozenset({κ, v, π, Nu, m, γ, Ea})",11,7,4,"[κ, v, π, Nu, m, γ, Ea]","[γ, κ, Ea, Gu, Nu, Pp, Pu, Pz, Vo, m, v]"
2327,NaN,lem,antiphonum,"[#Ea, #Gu, #Ma, #Nu, #Pp, #Pq, #Pu, #Pz, #Vm, ...","frozenset({Ma, π, Pq, Vm, Nu, Ea})",10,6,4,"[Ma, π, Pq, Vm, Nu, Ea]","[Ea, Gu, Ma, Nu, Pp, Pq, Pu, Pz, Vm, Vo]"
7413,NaN,lem,duabus,"[#β, #γ, #λ, #Ea, #Pp, #Pu, #Pz, #Vo, #m, #v]","frozenset({Pu, Pz, λ, Vo, Pp, v, m, α})",10,8,2,"[Pu, Pz, λ, Vo, Pp, v, m, α]","[β, γ, λ, Ea, Pp, Pu, Pz, Vo, m, v]"
253,NaN,lem,ysidorum,"[#β, #Gu, #Ma, #Pp, #Pu, #Px, #Pz, #Vb, #Vm, #Vo]","frozenset({Ma, π, Vm, β, Px, Vb})",10,6,4,"[Ma, π, Vm, β, Px, Vb]","[β, Gu, Ma, Pp, Pu, Px, Pz, Vb, Vm, Vo]"
406,NaN,lem,consueto,"[#α, #Gu, #Pp, #Pu, #Px, #Pz, #Vo, #m, #v]","frozenset({v, π, m, Px, α})",9,5,4,"[v, π, m, Px, α]","[α, Gu, Pp, Pu, Px, Pz, Vo, m, v]"
3695,NaN,lem,648.,"[#α, #Gu, #Pp, #Pu, #Pz, #Vo, #m, #v]","frozenset({v, π, α, m})",8,4,4,"[v, π, α, m]","[α, Gu, Pp, Pu, Pz, Vo, m, v]"
2581,NaN,lem,vocibus et,"[#α, #Gu, #Pp, #Pu, #Pz, #Vo, #m, #v]","frozenset({v, π, α, m})",8,4,4,"[v, π, α, m]","[α, Gu, Pp, Pu, Pz, Vo, m, v]"


In [68]:
candidates_reduced["proposed_sorted"] = candidates_reduced["proposed"].apply(lambda x: sorted(x))

In [69]:
candidates_reduced

,app_id,type,reading,witnesses,optimal_representation,actual_n_sigla,optimal_n_sigla,sigla_saved,proposed,witness_no_hash,proposed_sorted
126,NaN,lem,mundanam,"[#α, #Gu, #Mb, #Pp, #Pu, #Px, #Pz, #Vm, #Vo, #...","frozenset({Px, Mb, Vz, v, π, Vm, m, α})",12,8,4,"[Px, Mb, Vz, v, π, Vm, m, α]","[α, Gu, Mb, Pp, Pu, Px, Pz, Vm, Vo, Vz, m, v]","[Mb, Px, Vm, Vz, m, v, α, π]"
2282,NaN,lem,dyaphonia,"[#Ea, #Gu, #Ma, #Nu, #Pp, #Pu, #Pz, #Px, #Va, ...","frozenset({Px, Ea, v, Ma, π, m, Nu, Va})",12,8,4,"[Px, Ea, v, Ma, π, m, Nu, Va]","[Ea, Gu, Ma, Nu, Pp, Pu, Pz, Px, Va, Vo, m, v]","[Ea, Ma, Nu, Px, Va, m, v, π]"
2139,NaN,lem,netem,"[#γ, #Ea, #Gu, #Pp, #Pu, #Px, #Pz, #Vm, #Vo, #...","frozenset({γ, Px, Ea, Vz, v, π, m, Vm})",12,8,4,"[γ, Px, Ea, Vz, v, π, m, Vm]","[γ, Ea, Gu, Pp, Pu, Px, Pz, Vm, Vo, Vz, m, v]","[Ea, Px, Vm, Vz, m, v, γ, π]"
696,NaN,lem,notam,"[#γ, #κ, #Ea, #Gu, #Nu, #Pp, #Pu, #Pz, #Vo, #m...","frozenset({κ, v, π, Nu, m, γ, Ea})",11,7,4,"[κ, v, π, Nu, m, γ, Ea]","[γ, κ, Ea, Gu, Nu, Pp, Pu, Pz, Vo, m, v]","[Ea, Nu, m, v, γ, κ, π]"
2327,NaN,lem,antiphonum,"[#Ea, #Gu, #Ma, #Nu, #Pp, #Pq, #Pu, #Pz, #Vm, ...","frozenset({Ma, π, Pq, Vm, Nu, Ea})",10,6,4,"[Ma, π, Pq, Vm, Nu, Ea]","[Ea, Gu, Ma, Nu, Pp, Pq, Pu, Pz, Vm, Vo]","[Ea, Ma, Nu, Pq, Vm, π]"
7413,NaN,lem,duabus,"[#β, #γ, #λ, #Ea, #Pp, #Pu, #Pz, #Vo, #m, #v]","frozenset({Pu, Pz, λ, Vo, Pp, v, m, α})",10,8,2,"[Pu, Pz, λ, Vo, Pp, v, m, α]","[β, γ, λ, Ea, Pp, Pu, Pz, Vo, m, v]","[Pp, Pu, Pz, Vo, m, v, α, λ]"
253,NaN,lem,ysidorum,"[#β, #Gu, #Ma, #Pp, #Pu, #Px, #Pz, #Vb, #Vm, #Vo]","frozenset({Ma, π, Vm, β, Px, Vb})",10,6,4,"[Ma, π, Vm, β, Px, Vb]","[β, Gu, Ma, Pp, Pu, Px, Pz, Vb, Vm, Vo]","[Ma, Px, Vb, Vm, β, π]"
406,NaN,lem,consueto,"[#α, #Gu, #Pp, #Pu, #Px, #Pz, #Vo, #m, #v]","frozenset({v, π, m, Px, α})",9,5,4,"[v, π, m, Px, α]","[α, Gu, Pp, Pu, Px, Pz, Vo, m, v]","[Px, m, v, α, π]"
3695,NaN,lem,648.,"[#α, #Gu, #Pp, #Pu, #Pz, #Vo, #m, #v]","frozenset({v, π, α, m})",8,4,4,"[v, π, α, m]","[α, Gu, Pp, Pu, Pz, Vo, m, v]","[m, v, α, π]"
2581,NaN,lem,vocibus et,"[#α, #Gu, #Pp, #Pu, #Pz, #Vo, #m, #v]","frozenset({v, π, α, m})",8,4,4,"[v, π, α, m]","[α, Gu, Pp, Pu, Pz, Vo, m, v]","[m, v, α, π]"


In [70]:
candidates_reduced["witness_no_hash_sorted"] = candidates_reduced["witness_no_hash"].apply(lambda x: sorted(x))

In [71]:
candidates_reduced

,app_id,type,reading,witnesses,optimal_representation,actual_n_sigla,optimal_n_sigla,sigla_saved,proposed,witness_no_hash,proposed_sorted,witness_no_hash_sorted
126,NaN,lem,mundanam,"[#α, #Gu, #Mb, #Pp, #Pu, #Px, #Pz, #Vm, #Vo, #...","frozenset({Px, Mb, Vz, v, π, Vm, m, α})",12,8,4,"[Px, Mb, Vz, v, π, Vm, m, α]","[α, Gu, Mb, Pp, Pu, Px, Pz, Vm, Vo, Vz, m, v]","[Mb, Px, Vm, Vz, m, v, α, π]","[Gu, Mb, Pp, Pu, Px, Pz, Vm, Vo, Vz, m, v, α]"
2282,NaN,lem,dyaphonia,"[#Ea, #Gu, #Ma, #Nu, #Pp, #Pu, #Pz, #Px, #Va, ...","frozenset({Px, Ea, v, Ma, π, m, Nu, Va})",12,8,4,"[Px, Ea, v, Ma, π, m, Nu, Va]","[Ea, Gu, Ma, Nu, Pp, Pu, Pz, Px, Va, Vo, m, v]","[Ea, Ma, Nu, Px, Va, m, v, π]","[Ea, Gu, Ma, Nu, Pp, Pu, Px, Pz, Va, Vo, m, v]"
2139,NaN,lem,netem,"[#γ, #Ea, #Gu, #Pp, #Pu, #Px, #Pz, #Vm, #Vo, #...","frozenset({γ, Px, Ea, Vz, v, π, m, Vm})",12,8,4,"[γ, Px, Ea, Vz, v, π, m, Vm]","[γ, Ea, Gu, Pp, Pu, Px, Pz, Vm, Vo, Vz, m, v]","[Ea, Px, Vm, Vz, m, v, γ, π]","[Ea, Gu, Pp, Pu, Px, Pz, Vm, Vo, Vz, m, v, γ]"
696,NaN,lem,notam,"[#γ, #κ, #Ea, #Gu, #Nu, #Pp, #Pu, #Pz, #Vo, #m...","frozenset({κ, v, π, Nu, m, γ, Ea})",11,7,4,"[κ, v, π, Nu, m, γ, Ea]","[γ, κ, Ea, Gu, Nu, Pp, Pu, Pz, Vo, m, v]","[Ea, Nu, m, v, γ, κ, π]","[Ea, Gu, Nu, Pp, Pu, Pz, Vo, m, v, γ, κ]"
2327,NaN,lem,antiphonum,"[#Ea, #Gu, #Ma, #Nu, #Pp, #Pq, #Pu, #Pz, #Vm, ...","frozenset({Ma, π, Pq, Vm, Nu, Ea})",10,6,4,"[Ma, π, Pq, Vm, Nu, Ea]","[Ea, Gu, Ma, Nu, Pp, Pq, Pu, Pz, Vm, Vo]","[Ea, Ma, Nu, Pq, Vm, π]","[Ea, Gu, Ma, Nu, Pp, Pq, Pu, Pz, Vm, Vo]"
7413,NaN,lem,duabus,"[#β, #γ, #λ, #Ea, #Pp, #Pu, #Pz, #Vo, #m, #v]","frozenset({Pu, Pz, λ, Vo, Pp, v, m, α})",10,8,2,"[Pu, Pz, λ, Vo, Pp, v, m, α]","[β, γ, λ, Ea, Pp, Pu, Pz, Vo, m, v]","[Pp, Pu, Pz, Vo, m, v, α, λ]","[Ea, Pp, Pu, Pz, Vo, m, v, β, γ, λ]"
253,NaN,lem,ysidorum,"[#β, #Gu, #Ma, #Pp, #Pu, #Px, #Pz, #Vb, #Vm, #Vo]","frozenset({Ma, π, Vm, β, Px, Vb})",10,6,4,"[Ma, π, Vm, β, Px, Vb]","[β, Gu, Ma, Pp, Pu, Px, Pz, Vb, Vm, Vo]","[Ma, Px, Vb, Vm, β, π]","[Gu, Ma, Pp, Pu, Px, Pz, Vb, Vm, Vo, β]"
406,NaN,lem,consueto,"[#α, #Gu, #Pp, #Pu, #Px, #Pz, #Vo, #m, #v]","frozenset({v, π, m, Px, α})",9,5,4,"[v, π, m, Px, α]","[α, Gu, Pp, Pu, Px, Pz, Vo, m, v]","[Px, m, v, α, π]","[Gu, Pp, Pu, Px, Pz, Vo, m, v, α]"
3695,NaN,lem,648.,"[#α, #Gu, #Pp, #Pu, #Pz, #Vo, #m, #v]","frozenset({v, π, α, m})",8,4,4,"[v, π, α, m]","[α, Gu, Pp, Pu, Pz, Vo, m, v]","[m, v, α, π]","[Gu, Pp, Pu, Pz, Vo, m, v, α]"
2581,NaN,lem,vocibus et,"[#α, #Gu, #Pp, #Pu, #Pz, #Vo, #m, #v]","frozenset({v, π, α, m})",8,4,4,"[v, π, α, m]","[α, Gu, Pp, Pu, Pz, Vo, m, v]","[m, v, α, π]","[Gu, Pp, Pu, Pz, Vo, m, v, α]"


In [72]:
candidates_reduced.columns

Index(['app_id', 'type', 'reading', 'witnesses', 'optimal_representation',
       'actual_n_sigla', 'optimal_n_sigla', 'sigla_saved', 'proposed',
       'witness_no_hash', 'proposed_sorted', 'witness_no_hash_sorted'],
      dtype='str')

In [76]:
final_candidates = candidates_reduced[
    [
        "type",
        "reading",
        "witnesses",
        "witness_no_hash_sorted",
        "proposed_sorted",
        "actual_n_sigla",
        "optimal_n_sigla",
        "sigla_saved"
    ]
]

In [77]:
final_candidates = final_candidates.rename(columns={
    "witness_no_hash_sorted": "witnesses",
    "proposed_sorted": "proposed",
    "actual_n_sigla": "actual_sigla",
    "optimal_n_sigla": "optimal_sigla",
    "sigla_saved": "difference"
})

In [78]:
import csv
final_candidates.to_csv("candidates.csv", index=False, quoting=csv.QUOTE_ALL)